# DiffSinger Miadu Fine-tuning (100步高頻保存版)
**修正內容**：
- **高頻保存**：每 100 步保存一次權重，防止 Colab 斷線丟進度。
- **斷點續練**：重啟後自動找回最新進度。
- **自動更新**：每次啟動會自動獲取最新腳本修正。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取/更新代碼與環境配置

In [ ]:
%cd /content/
import os
if not os.path.exists('diffsinger-repo'):
    !git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
else:
    print('| 偵測到已有代碼，正在執行更新...')
    %cd diffsinger-repo
    !git pull
    %cd ..

%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第三步：數據同步與斷點檢查

In [ ]:
import os
import torch

# GPU 檢查
gpu_ok = torch.cuda.is_available()
print(f'| GPU 狀態: {"✅ 已就緒" if gpu_ok else "❌ 未偵測到 GPU"}')

drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"
print(f'| 使用 Drive 目錄: {drive_root}')

# Checkpoints 軟連結 (斷點續練核心)
!mkdir -p "{drive_root}/checkpoints_finetune"
if not os.path.islink("checkpoints"):
    !rm -rf checkpoints
    !ln -s "{drive_root}/checkpoints_finetune" checkpoints

def robust_sync(src_dir, dest_path, min_size_mb=1):
    dest_dir = os.path.dirname(dest_path)
    !mkdir -p "{dest_dir}"
    if not os.path.exists(dest_path) or os.path.getsize(dest_path) < min_size_mb * 1024 * 1024:
        print(f"| 正在同步: {src_dir}")
        !cp -rv "{src_dir}" "{dest_dir}/.."
    else:
        print(f"| 已跳過完整檔案: {os.path.basename(dest_path)}")

robust_sync(f"{drive_root}/1117_opencpop_ds1000_strict_pinyin", 
            "checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt", 800)
robust_sync(f"{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02", 
            "checkpoints/nsf_hifigan_44.1k_hop512_128bin_2024.02/model_ckpt_steps_160000.ckpt", 10)
robust_sync(f"{drive_root}/miadu_finetune", 
            "data/binary/miadu_finetune/train.data", 1000)

print('| 數據準備完成！已自動連接 Drive 進度。')

## 第四步：啟動訓練

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1